# 14_run_docking — smina로 도킹 실행 & 랭킹

**한 줄 요약:** 준비한 리간드들을 단백질 포켓에 **도킹**(끼워 맞춰 결합에너지 계산)하고, 대조군(공결정 저해제·BI-3231)과 비교해 순위를 매긴다.
**용어:** 도킹=리간드를 단백질 포켓에 넣어보는 계산 / 결합에너지(kcal/mol)=낮을수록 강한 결합 / smina=도킹 프로그램.
**큰 흐름:** ① 준비·함수 → ② 이전 결과 재사용(resume) → ③ 대조군1 → ④ 대조군2(BI-3231) → ⑤ 후보 도킹 → ⑥ 랭킹·저장

> **📌 읽는 법**: 각 코드 셀은 **[① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기]**. ③은 그 셀에 **처음 나온** 함수·문법(기초 반복은 *(01에서 설명)*).

### 준비 — 폴더 위치 맞추기
어느 폴더에서 열어도 프로젝트 최상위에서 실행되도록 이동.

In [ ]:
# 노트북을 어느 폴더에서 열든 프로젝트 루트에서 실행되도록 이동
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())

🔎 *(01에서 설명)*: `os.chdir('..')`=상위 폴더 이동, `print`=출력.

### 셀 1 — 도구 + smina 확인 + 도킹 함수
라이브러리를 가져오고 smina가 설치됐는지 확인한 뒤, 한 분자를 도킹하는 함수를 만든다.

In [ ]:
import os
import re
import sys
import glob
import shutil
import tempfile
import subprocess
import urllib.request
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

RECDIR = "data/docking/receptor"
REC_PDBQT = os.path.join(RECDIR, "receptor.pdbqt")
REF_LIG = os.path.join(RECDIR, "ref_ligand.pdb")
LIGANDS = "data/docking/hsd17b13_ligands.sdf"
MANIFEST = "data/docking/docking_manifest.csv"
RESDIR = "data/docking/results"
OUT_CSV = os.path.join(RESDIR, "docking_scores.csv")
EXHAUST = 8          # 정확도(↑느림). 빠른 테스트는 4
AUTOBOX_ADD = 4      # 기준 리간드 경계 + Å
os.makedirs(RESDIR, exist_ok=True)


def which(name):
    return os.environ.get("SMINA") if name == "smina" and os.environ.get("SMINA") \
        else (shutil.which(name) or shutil.which(name + ".exe"))


SMINA = which("smina")
if not SMINA:
    sys.exit("[중단] smina 미설치. 설치 후 재실행:\n"
             "  conda install -c conda-forge smina\n"
             "  (또는 공식 smina.exe를 PATH에 두거나 환경변수 SMINA=경로 설정)")
if not (os.path.exists(REC_PDBQT) and os.path.exists(REF_LIG)):
    sys.exit("[중단] 수용체 준비 안 됨. 먼저: python scripts/13_prep_receptor.py")
print(f"smina: {SMINA}")


def run_smina(ligand_path, out_path):
    """도킹 실행 → 최고 모드 결합에너지(kcal/mol) 반환(실패 시 None)"""
    cmd = [SMINA, "-r", REC_PDBQT, "-l", ligand_path,
           "--autobox_ligand", REF_LIG, "--autobox_add", str(AUTOBOX_ADD),
           "--exhaustiveness", str(EXHAUST), "--seed", "42",
           "-o", out_path, "--cpu", "0"]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=1200)
    except subprocess.TimeoutExpired:
        return None
    best = None
    for line in r.stdout.splitlines():
        m = re.match(r"\s*1\s+(-?\d+\.\d+)", line)   # mode 1 = 최고
        if m:
            best = float(m.group(1))
            break
    return best


def sdf_iter(path):
    for mol in Chem.SDMolSupplier(path, removeHs=False):
        if mol is not None:
            yield mol


def dock_mol(mol, tag):
    with tempfile.NamedTemporaryFile("w", suffix=".sdf", delete=False) as tf:
        tmp = tf.name
    w = Chem.SDWriter(tmp)
    w.write(mol)
    w.close()
    out = os.path.join(RESDIR, f"pose_{tag}.pdbqt")
    aff = run_smina(tmp, out)
    os.unlink(tmp)
    return aff


man = pd.read_csv(MANIFEST).set_index("np_id") if os.path.exists(MANIFEST) else None
rows = []

🔎 **코드 뜯어보기 (셀 1)**
- `SMINA = which("smina")` : smina 프로그램 위치 찾기. 없으면 `sys.exit(...)`로 안내 후 중단.
- `def run_smina(ligand_path, out_path):` : 리간드 하나를 도킹하는 함수. `subprocess.run([SMINA, "-r", ...], ...)`=smina 실행.
- `re.match(r"\s*1\s+(-?\d+\.\d+)", line)` : 출력에서 **1번 모드의 결합에너지 숫자**를 정규식으로 뽑기(-?=음수 가능).
- `Chem.SDMolSupplier(path)` : SDF에서 분자들을 하나씩 읽는 도구.

### 셀 2 — 이미 한 결과 재사용(resume) 준비
예전에 저장한 결과가 있으면 다시 계산하지 않도록 불러오고, 물질명→SMILES 조회 함수를 만든다.

In [ ]:
done = {}
if os.path.exists(OUT_CSV):
    prev = pd.read_csv(OUT_CSV)
    for _, pr in prev.iterrows():
        if pd.notna(pr.get("affinity")):
            done[str(pr["id"])] = pr.to_dict()
    if done:
        print(f"(resume) 기존 결과 {len(done)}건 재사용, 빠진 것만 도킹")


def pubchem_smiles(name):
    """PubChem 이름→SMILES. 속성명 변경(IsomericSMILES→SMILES) 대응."""
    import json
    for prop in ("SMILES", "ConnectivitySMILES", "IsomericSMILES", "CanonicalSMILES"):
        try:
            u = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
                 f"{name}/property/{prop}/JSON")
            with urllib.request.urlopen(u, timeout=30) as resp:
                d = json.load(resp)["PropertyTable"]["Properties"][0]
            for k in (prop, "SMILES", "ConnectivitySMILES", "IsomericSMILES", "CanonicalSMILES"):
                if d.get(k):
                    return d[k]
        except Exception:
            continue
    return None

🔎 **코드 뜯어보기 (셀 2)**
- `done = {}` : 이미 도킹한 결과를 담을 딕셔너리. 파일이 있으면 읽어와 채운다(**resume**=이어서 하기).
- `def pubchem_smiles(name):` : 물질 이름으로 PubChem에서 SMILES를 받아오는 함수. 속성명이 바뀔 수 있어 여러 이름을 순서대로 시도(`for prop in (...)`).

### 셀 3 — 대조군 1: 공결정 저해제 다시 도킹
원래 결합해 있던 저해제를 도킹해 **기준선(양성 대조)** 을 만든다.

In [ ]:
if "REF_cocrystal" in done:
    rows.append(done["REF_cocrystal"])
    print(f"\n[대조군] 공결정 저해제: {done['REF_cocrystal']['affinity']} kcal/mol (재사용)")
else:
    print("\n[대조군] 공결정 저해제 redocking...")
    aff = run_smina(REF_LIG, os.path.join(RESDIR, "pose_REF_redock.pdbqt"))
    print(f"  공결정 저해제: {aff} kcal/mol (양성 기준선)")
    rows.append({"id": "REF_cocrystal", "type": "control", "affinity": aff})

🔎 **코드 뜯어보기 (셀 3)**
- `if "REF_cocrystal" in done:` : 이미 했으면 재사용, 아니면 `run_smina(...)`로 도킹. `rows.append({...})`=결과를 표에 추가.

### 셀 4 — 대조군 2: BI-3231 (알려진 저해제)
PubChem에서 BI-3231을 받아 도킹해 두 번째 기준선을 만든다.

In [ ]:
if "BI-3231" in done:
    rows.append(done["BI-3231"])
    print(f"[대조군] BI-3231: {done['BI-3231']['affinity']} kcal/mol (재사용)")
else:
    print("[대조군] BI-3231 조회(PubChem)...")
    try:
        smi = pubchem_smiles("BI-3231")
        if not smi:
            raise ValueError("PubChem에서 SMILES 못 찾음")
        m = Chem.AddHs(Chem.MolFromSmiles(smi))
        p = AllChem.ETKDGv3(); p.randomSeed = 42
        AllChem.EmbedMolecule(m, p); AllChem.MMFFOptimizeMolecule(m)
        m.SetProp("_Name", "BI-3231")
        aff = dock_mol(m, "BI3231")
        print(f"  BI-3231: {aff} kcal/mol (알려진 저해제 기준선)")
        rows.append({"id": "BI-3231", "type": "control", "affinity": aff})
    except Exception as e:
        print(f"  BI-3231 조회/도킹 실패(건너뜀): {e}")

🔎 **코드 뜯어보기 (셀 4)**
- `smi = pubchem_smiles("BI-3231")` : 이름으로 SMILES 조회. 이후 3D로 만들어 도킹(12에서 본 AddHs/Embed/MMFF 사용). 실패하면 `except`로 건너뜀.

### 셀 5 — 후보 리간드 도킹
준비한 후보들을 하나씩 도킹해 결합에너지를 구한다.

In [ ]:
print(f"\n[후보] {LIGANDS} 도킹 시작 (exhaustiveness={EXHAUST})")
mols = list(sdf_iter(LIGANDS))
for i, mol in enumerate(mols, 1):
    npid = mol.GetProp("np_id") if mol.HasProp("np_id") else mol.GetProp("_Name")
    if npid in done:
        rows.append(done[npid])
        print(f"  [{i}/{len(mols)}] {npid}: {done[npid]['affinity']} kcal/mol (재사용)")
        continue
    aff = dock_mol(mol, npid)
    prob = mol.GetProp("active_prob") if mol.HasProp("active_prob") else ""
    sim = mol.GetProp("max_sim_known") if mol.HasProp("max_sim_known") else ""
    rows.append({"id": npid, "type": "candidate", "affinity": aff,
                 "active_prob": prob, "max_sim_known": sim})
    print(f"  [{i}/{len(mols)}] {npid}: {aff} kcal/mol")

🔎 **코드 뜯어보기 (셀 5)**
- `for i, mol in enumerate(mols, 1):` : 후보 분자를 1번부터 반복. 이미 한 것은 건너뛰고, 새 것만 `dock_mol(...)`로 도킹.

### 셀 6 — 결과 랭킹·저장
모든 점수를 모아 강한 순으로 정렬하고, 대조군 기준선을 넘는 후보를 표시해 저장한다.

In [ ]:
res = pd.DataFrame(rows)
res["affinity"] = pd.to_numeric(res["affinity"], errors="coerce")
res = res.sort_values("affinity").reset_index(drop=True)  # 낮을수록 강함
res.to_csv(OUT_CSV, index=False)

ctrl = res[res.type == "control"]["affinity"]
ref_score = ctrl.max() if len(ctrl) else None   # 가장 약한 대조군 기준선
print(f"\n결과 저장: {OUT_CSV}")
print("\n=== 결합에너지 랭킹(낮을수록 강함) ===")
print(res[["id", "type", "affinity", "active_prob", "max_sim_known"]].to_string(index=False))
if ref_score is not None:
    good = res[(res.type == "candidate") & (res.affinity <= ref_score)]
    print(f"\n대조군 기준선({ref_score:.1f}) 이상으로 결합한 후보: {len(good)}개")
    print("→ 이들만 포즈(PyMOL)로 상호작용 확인 후 실험 후보로.")

🔎 **코드 뜯어보기 (셀 6)**
- `res = pd.DataFrame(rows)` : 점수들을 표로. `pd.to_numeric(..., errors="coerce")`=숫자로(실패는 빈 값).
- `.sort_values("affinity")` : 결합에너지 **오름차순**(낮을수록 강해서 위로).
- `res[(res.type=="candidate") & (res.affinity <= ref_score)]` : 대조군 기준선 이하로 결합한 후보만 추리기.